In [1]:
# Create an API Client

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

In [2]:
# Building Helper Functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content":text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Messages with Default Responses
messages = []

add_user_message(messages, "Generate a very short event bridge rule as json")

chat(messages)

'Here\'s a simple EventBridge rule in JSON:\n\n```json\n{\n  "Name": "my-rule",\n  "EventPattern": {\n    "source": ["aws.ec2"],\n    "detail-type": ["EC2 Instance State-change Notification"],\n    "detail": {\n      "state": ["stopped"]\n    }\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Id": "my-target",\n      "Arn": "arn:aws:sns:us-east-1:123456789012:my-topic"\n    }\n  ]\n}\n```\n\nThis rule:\n- **Triggers** when an EC2 instance is **stopped**\n- **Sends** a notification to an **SNS topic**'

In [ ]:
# Messages with Structured Response

# Since pre-filling is not supported in Claude, 
# we can use the system prompt to instruct the model to only respond with json 
# and not include any additional text.

messages = []

system = """
You are an EventBridge Rule Generator.
You will generate json format but in the response exclude ```json at the start and at the end.
You will provide only the json responde body without any additional text."""

add_user_message(messages, "Generate a very short event bridge rule as json")
# add_assistant_message(messages, "```json")

text = chat(messages, system)

text

'{\n  "source": ["aws.ec2"],\n  "detail-type": ["EC2 Instance State-change Notification"],\n  "detail": {\n    "state": ["stopped"]\n  }\n}'

In [12]:
import json

clean_json = json.loads(text.strip())

In [13]:
clean_json

{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['stopped']}}